# Getting started with CausalDemand

This notebook walks through one complete run of the benchmark on the facial tissue category. We download the 20
datasets, fit a simple model, write its predictions in the format that `causaldemand score` expects, and score them
against the reference models. The download needs about 14 GB of disk;
the rest of the notebook runs in about two minutes.

Before you start, install the package with `pip install causaldemand`, accept the access conditions on the Hugging
Face dataset page, and set `HF_TOKEN` to an access token of your Hugging Face account.

## Step 1: Download the data

The facial tissue category has 20 datasets: two demand models (discrete-choice and log-linear), each with confounding
off and on, at five seeds. The 20 datasets are stored in sub-folders under `causaldemand_data/`. A method may use only
the files in `public/`.

In [ ]:
!causaldemand download tissue | grep -E "^\[|^Done"

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

DATA = Path("causaldemand_data")
PAIR = ["product_id", "store_id"]
KEY = PAIR + ["week"]
TEXT = {"product_id": str, "store_id": str, "intervention_id": str}
INSTRUMENTS = ["promo_cost", "log_cost"]


def read(folder, name):
    return pd.read_csv(folder / "public" / f"{name}.csv", dtype=TEXT)


folders = sorted(p for p in DATA.glob("tissue_*") if (p / "public").is_dir())
[f.name for f in folders]

## Step 2: Load the training data

The training file has one row for each product, store and week with positive sales, over 140 weeks. Two of its columns
are instruments: `promo_cost` sets how deep a discount is, and `supply_cost_proxy` moves the regular price.

In [ ]:
train = read(DATA / "tissue_discrete-choice_on_seed1", "transactions_train_public")
train.head()

## Step 3: Define the model

We model the logarithm of units sold as a linear function of the logarithm of price: log(units) = a + b log(price).
Each product-store pair has its own intercept a, and each dataset has one slope b, the price elasticity: the
percentage change in units sold when the price rises by 1%.

- **Confounding off:** prices are set independently of demand, so fitting b by ordinary least squares (OLS) gives the
  right answer.
- **Confounding on:** stores give deeper discounts in weeks when demand is already high, so low prices and high sales
  occur together for a reason other than the price itself, and the OLS slope comes out too steep. Two-stage least
  squares (2SLS) gives the right answer here. The two instrument columns change prices for reasons unrelated to demand;
  2SLS first predicts log price from the instruments and then fits b on that predicted price, so it uses only the
  price changes that the instruments cause.

In [ ]:
def fit(train, instruments=True):
    t = train.assign(log_units=np.log(train["units"]), log_price=np.log(train["price"]),
                     log_cost=np.log(train["supply_cost_proxy"]))
    by_pair = t.groupby(PAIR)
    within = {c: t[c] - by_pair[c].transform("mean") for c in ["log_units", "log_price"] + INSTRUMENTS}
    if instruments:
        z = np.column_stack([within[c] for c in INSTRUMENTS])
        price = z @ np.linalg.lstsq(z, within["log_price"], rcond=None)[0]
    else:
        price = within["log_price"]
    elasticity = float((price * within["log_units"]).sum() / (price * within["log_price"]).sum())
    level = (by_pair["log_units"].mean() - elasticity * by_pair["log_price"].mean()).rename("level")
    return elasticity, level.reset_index()

## Step 4: Compare OLS and 2SLS under confounding

The confounding-off and confounding-on versions of a dataset share every random draw except the discounts, so their
true elasticity is the same. Here we compare the two estimators on the discrete-choice datasets with seed 1. With
confounding on, the OLS estimate moves from -1.47 to -1.83, while the 2SLS estimate stays at -1.41.

In [ ]:
rows = []
for setting in ("off", "on"):
    t = read(DATA / f"tissue_discrete-choice_{setting}_seed1", "transactions_train_public")
    rows.append({"confounding": setting, "OLS": fit(t, instruments=False)[0], "2SLS": fit(t)[0]})
pd.DataFrame(rows).set_index("confounding").round(3)

## Step 5: Write predictions for all 20 datasets

`causaldemand score` reads two files for each dataset: `forecast.csv`, with the predicted units for every row of the
evaluation file, and `scenarios.csv`, with the predicted change in units for every scenario row where the price
changes. The table below shows the estimated elasticity of each dataset; the confounding-on and confounding-off
estimates agree at every seed.

In [ ]:
def predict(rows, price, elasticity, level):
    lv = rows[PAIR].merge(level, on=PAIR, how="left")["level"].to_numpy()
    return np.exp(lv + elasticity * np.log(rows[price].to_numpy()))


def write(folder, out):
    elasticity, level = fit(read(folder, "transactions_train_public"))
    holdout = read(folder, "transactions_holdout_context_public")
    forecast = holdout[KEY].assign(predicted_units=predict(holdout, "price", elasticity, level))
    panel = read(folder, "counterfactual_sweep_context_panel")
    moved = panel[panel["intervention_price"] != panel["baseline_price"]]
    new, old = (predict(moved, p, elasticity, level) for p in ("intervention_price", "baseline_price"))
    scenarios = moved[["intervention_id"] + KEY].assign(predicted_delta_units=new - old)
    target = out / folder.name
    target.mkdir(parents=True, exist_ok=True)
    forecast.to_csv(target / "forecast.csv", index=False)
    scenarios.to_csv(target / "scenarios.csv", index=False)
    return elasticity


estimates = pd.Series({f.name: write(f, Path("baseline")) for f in folders}, name="elasticity")
parts = estimates.index.str.extract(r"tissue_(.+)_(on|off)_seed(\d+)")
pd.DataFrame({"demand model": parts[0].values, "confounding": parts[1].values, "seed": parts[2].astype(int).values,
              "elasticity": estimates.values}).pivot_table(index="seed", columns=["demand model", "confounding"],
                                                         values="elasticity").round(3)

## Step 6: Score the predictions

`causaldemand score` compares the predictions with the answer keys and prints the results next to the reference
models: the counterfactual WMAPE over the 32 price scenarios and the forecast WMAPE, each averaged over the five seeds
with a 95% interval.

In [ ]:
!causaldemand score tissue baseline

## Next steps

To evaluate your own method, replace `fit` and `predict` and run the notebook again.